# Capítulo 3 · Actividad: análisis de datos con NumPy, pandas y matplotlib

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/01_actividad_numpy_pandas.ipynb)

En esta actividad haces de **analista de datos**: vas a responder preguntas sobre datos reales con **NumPy** y **pandas**, y después los visualizarás con **matplotlib** y **seaborn**. Lo importante no es solo que el código funcione, sino que **sepas interpretar los resultados**.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia. Si editas el original tendrás conflictos al sincronizar tu fork.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```
3. Ten a mano las notebooks de clase de `notebooks/lectures/chapter_03/`: `01_numpy`, `02_pandas` y `03_matplotlib`.

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Puedes añadir todas las celdas que necesites.
- En las celdas ✍️ responde con tus palabras y **apoya tus respuestas con números** de tus resultados.
- En las partes 1 y 2, las 💡 **pistas** te recuerdan funciones. Si una función no salió en clase, la pista explica cómo usarla. La parte 3 no tiene pistas.
- Al terminar, ejecuta **Kernel → Restart & Run All** y comprueba que todo funciona de arriba abajo.

| Parte | Datos | Herramienta |
|---|---|---|
| 1. La cafetería | ventas de un mes (generadas en la notebook) | NumPy |
| 2. Las propinas | `data/tips.csv` | pandas (+ un poco de NumPy) |
| 3. Visualización | cafetería, propinas y `data/penguins.csv` | matplotlib y seaborn |

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

Carpeta de datos: /content/data


---
## Parte 1 · La cafetería ☕ (NumPy)

Una cafetería vende 5 productos y ha registrado cuántas unidades vendió de cada uno durante **4 semanas** (28 días, de lunes a domingo). La dueña quiere saber cuánto gana, qué días van mejor y qué producto le conviene más.

Ejecuta la siguiente celda para generar los datos. **No hace falta que la modifiques ni que entiendas cada línea.**

In [4]:
# Datos de la cafetería (no hace falta modificar esta celda)
rng = np.random.default_rng(42)

productos = np.array(["café", "latte", "té", "croissant", "sándwich"])
precios = np.array([1.5, 2.8, 1.2, 2.0, 4.5])       # precio de cada producto, en $

# Unidades vendidas cada día: los fines de semana se vende un 50% más
dia_semana = np.tile(np.arange(7), 4)                # 0 = lunes, ..., 6 = domingo
factor = np.where(dia_semana >= 5, 1.5, 1.0)
promedio = np.outer(factor, [30, 22, 12, 18, 9])     # ventas esperadas por día y producto
ventas_planas = rng.poisson(promedio).ravel()         # ¡todo en una sola fila!

print(ventas_planas[:10], "...")
print("Número de valores:", ventas_planas.size)

[35 28  7 16 13 29 16 10 18 11] ...
Número de valores: 140


### Ejercicio 1 · Dar forma a los datos

`ventas_planas` tiene 140 números seguidos: primero las ventas de los 5 productos del día 1, después las del día 2, y así sucesivamente.

a) Crea una matriz `ventas` con forma `(28, 5)`: una **fila por día** y una **columna por producto** (en el mismo orden que `productos`).

b) Imprime su `shape`, `ndim`, `size` y `dtype`.

c) ¿Cuántos cafés se vendieron el primer día? Comprueba que coincide con `ventas_planas[0]`.

> 💡 **Pista:** `reshape`.

In [7]:
# Tu código aquí
ventas = ventas_planas.reshape (28, 5)
print("shape:", ventas.shape)
print("ndim:", ventas.ndim)
print("size:", ventas.size)
print("dtype:", ventas.dtype)

cafes_primer_dia = ventas [0, 0]
print("cafes vendidos el primer dia (matriz):", cafes_primer_dia)
print("Coincide con ventas_planas[0]:", cafes_primer_dia== ventas_planas[0])


shape: (28, 5)
ndim: 2
size: 140
dtype: int64
cafes vendidos el primer dia (matriz): 35
Coincide con ventas_planas[0]: True


### Ejercicio 2 · Indexación

Obtén, usando índices y *slicing* sobre `ventas`:

a) Las ventas de todos los productos el **primer día**.

b) Las ventas de **latte** (columna 1) durante todo el mes.

c) Las ventas de la **última semana** (los últimos 7 días). Comprueba que su forma es `(7, 5)`.

d) Las ventas de **croissant y sándwich** (las dos últimas columnas) durante la **primera semana**.

> 💡 **Pista:** `M[filas, columnas]`. Recuerda que los índices negativos cuentan desde el final.

In [8]:
# a) ventas de todos los productos el primer día
ventas_primer_dia = ventas[0, :]
print("a) Ventas primer día:", ventas_primer_dia)

# b) ventas de latte (columna 1) durante todo el mes
ventas_latte = ventas[:, 1]
print("b) Ventas de latte todo el mes:", ventas_latte)

# c) ventas de la última semana (últimos 7 días) y comprobación de su forma
ultima_semana = ventas[-7:, :]
print("c) Ventas última semana:\n", ultima_semana)
print("   Forma (shape):", ultima_semana.shape)

# d) Las ventas de croissant y sándwich (últimas 2 columnas) durante la primera semana
croissant_sandwich_primera_semana = ventas[:7, -2:]
print("d) Croissant y sándwich primera semana:\n", croissant_sandwich_primera_semana)

a) Ventas primer día: [35 28  7 16 13]
b) Ventas de latte todo el mes: [28 16 17 21 17 29 37 16 23 25 22 19 29 39 24 19 18 22 30 44 27 14 31 21
 30 26 32 45]
c) Ventas última semana:
 [[22 14 11 20  9]
 [34 31 12 27  7]
 [26 21 17 23  7]
 [35 30  3  9 10]
 [23 26 16 21  8]
 [38 32 20 21 11]
 [49 45 18 27  9]]
   Forma (shape): (7, 5)
d) Croissant y sándwich primera semana:
 [[16 13]
 [18 11]
 [17 10]
 [15  8]
 [17  8]
 [22 14]
 [29 12]]


### Ejercicio 3 · Ingresos

a) Calcula la matriz `ingresos` (forma `(28, 5)`): lo que se ingresó por cada producto cada día.

b) ¿Cuánto ingresó la cafetería en todo el mes?

c) Crea `ingreso_diario`, un array con el ingreso total de **cada día** (forma `(28,)`).

d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?

> 💡 **Pistas:**
> - `ventas * precios` multiplica **cada fila** de `ventas` por el vector `precios`, elemento a elemento: la columna del café por 1.5, la del latte por 2.8… (este mecanismo se llama *broadcasting*).
> - `M.sum()` suma todo. `M.sum(axis=1)` suma **cada fila** y devuelve un resultado por fila; `M.sum(axis=0)` suma **cada columna**.

In [10]:
# a) Calcula la matriz ingresos (forma (28, 5))
ingresos = ventas * precios

# b) ¿Cuánto ingresó la cafetería en todo el mes?
ingresos_totales_mes = ingresos.sum()
print("b) Ingresos totales del mes: $", ingresos_totales_mes)

# c) Crea ingreso_diario, un array con el ingreso total de cada día (forma (28,))
ingreso_diario = ingresos.sum(axis=1)
print("c) Ingreso diario:", ingreso_diario)

# d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?
ingreso_medio = ingreso_diario.mean()
ingreso_max = ingreso_diario.max()
ingreso_min = ingreso_diario.min()

print("d)")
print("Ingreso diario medio: $", ingreso_medio)
print("Ingreso diario máximo: $", ingreso_max)
print("Ingreso diario mínimo: $", ingreso_min)

b) Ingresos totales del mes: $ 6342.5
c) Ingreso diario: [229.8 185.8 194.1 179.1 178.2 291.7 319.1 188.9 183.2 198.3 204.6 191.3
 320.7 317.5 181.9 174.1 176.6 196.8 229.  337.3 279.9 165.9 237.7 195.7
 203.1 204.5 262.1 315.6]
d)
Ingreso diario medio: $ 226.5178571428572
Ingreso diario máximo: $ 337.29999999999995
Ingreso diario mínimo: $ 165.89999999999998


### Ejercicio 4 · Días buenos y días flojos

Usa este array con el número de cada día (del 1 al 28):
```python
dias = np.arange(1, 29)
```

a) Crea una máscara con los días cuyo ingreso supera el **ingreso diario medio** y muestra **qué días** fueron.

b) ¿**Cuántos** días fueron?

c) Crea un array que diga `"bueno"` o `"flojo"` para cada día, según si supera o no la media.

d) Calcula `dias[mascara] % 7` (el resto de dividir entre 7). El día 1 fue lunes, así que un resto de **6** es sábado y un resto de **0** es domingo. ¿Qué tienen en común los días buenos?

> 💡 **Pistas:** `.sum()` sobre una máscara cuenta los `True`, porque `True` vale 1 y `False` vale 0. Para c), `np.where`.

In [11]:
dias =np.arange(1, 29)
mascara = ingreso_diario > ingreso_medio
dias_buenos = dias[mascara]

print ("a)")
print("Dias que los ingresos superan la media:", dias_buenos)

num_dias_buenos = mascara.sum()
print("b)")
print("Dias buenos:", num_dias_buenos)

etiqueta_dias = np.where(mascara, "bueno", "flojo")
print("c)")
print("Estado de cada dia:", etiqueta_dias)

resto_dias_buenos = dias[mascara] % 7
print("d)")
print("Restos de los dias buenos dividido entre 7:", resto_dias_buenos)

a)
Dias que los ingresos superan la media: [ 1  6  7 13 14 19 20 21 23 27 28]
b)
Dias buenos: 11
c)
Estado de cada dia: ['bueno' 'flojo' 'flojo' 'flojo' 'flojo' 'bueno' 'bueno' 'flojo' 'flojo'
 'flojo' 'flojo' 'flojo' 'bueno' 'bueno' 'flojo' 'flojo' 'flojo' 'flojo'
 'bueno' 'bueno' 'bueno' 'flojo' 'bueno' 'flojo' 'flojo' 'flojo' 'bueno'
 'bueno']
d)
Restos de los dias buenos dividido entre 7: [1 6 0 6 0 5 6 0 2 6 0]


### Ejercicio 5 · Análisis por semanas

a) Reorganiza `ingreso_diario` en una matriz `por_semana` de forma `(4, 7)`: una fila por semana y una columna por día (lunes a domingo).

b) ¿Cuánto se ingresó **cada semana**?

c) ¿Cuál es el ingreso medio de **cada día de la semana**? ¿Qué día se vende más? Usa:
```python
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
```

d) Compara el ingreso medio de un día de **fin de semana** con el de un día **entre semana**. ¿Cuántas veces más se ingresa en fin de semana?

> 💡 **Pistas:** `axis=1` da un resultado por fila (por semana) y `axis=0` uno por columna (por día de la semana). `arr.argmax()` devuelve la **posición** del valor máximo; úsala como índice en `nombres_dias`.

In [18]:
por_semana = ingreso_diario.reshape(4, 7)
print("a)")
print(" Matriz por_semana (4, 7):\n", por_semana)

ingreso_por_semana = por_semana.sum(axis=1)
print("b)")
print("Ingreso por semana:\n", ingreso_por_semana)

nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
ingreso_medio_por_dia = por_semana.mean(axis=0)

dia_mas_ventas_idx = ingreso_medio_por_dia.argmax()
dia_mas_ventas = nombres_dias[dia_mas_ventas_idx]

print("c)")
print("Ingreso medio por día de la semana:\n", ingreso_medio_por_dia)
print("   Día que más se vende:", dia_mas_ventas)


media_entre_semana = ingreso_medio_por_dia[:5].mean()
media_fin_de_semana = ingreso_medio_por_dia[5:].mean()
veces_mas = media_fin_de_semana / media_entre_semana

print("d)")
print("Ingreso medio entre semana:\n $", media_entre_semana)
print(f"   Ingreso medio en fin de semana:\n ${media_fin_de_semana:.2f}")
print(f"   En fin de semana se ingresa {veces_mas:.2f} veces más.")

a)
 Matriz por_semana (4, 7):
 [[229.8 185.8 194.1 179.1 178.2 291.7 319.1]
 [188.9 183.2 198.3 204.6 191.3 320.7 317.5]
 [181.9 174.1 176.6 196.8 229.  337.3 279.9]
 [165.9 237.7 195.7 203.1 204.5 262.1 315.6]]
b)
Ingreso por semana:
 [1577.8 1604.5 1575.6 1584.6]
c)
Ingreso medio por día de la semana:
 [191.625 195.2   191.175 195.9   200.75  302.95  308.025]
   Día que más se vende: domingo
d)
Ingreso medio entre semana:
 $ 194.93
   Ingreso medio en fin de semana:
 $305.49
   En fin de semana se ingresa 1.57 veces más.


### Ejercicio 6 · El producto estrella

a) Calcula el ingreso total de **cada producto** en el mes.

b) ¿Qué **porcentaje** del ingreso total aporta cada producto? Redondea a 1 decimal.

c) ¿Qué producto genera **más dinero**? ¿Y cuál vende **más unidades**?

> 💡 **Pista:** `axis=0` y `argmax`, igual que en el ejercicio anterior.

In [19]:
# a) ingreso total de cada producto en el mes
ingreso_por_producto = ingresos.sum(axis=0)

# b) ¿Qué porcentaje del ingreso total aporta cada producto? (Redondeado a 1 decimal)
porcentaje_ingresos = (ingreso_por_producto / ingresos_totales_mes) * 100
porcentaje_ingresos_redondeado = np.round(porcentaje_ingresos, 1)

# c) ¿Qué producto genera más dinero? ¿Y cuál vende más unidades?
producto_mas_dinero_idx = ingreso_por_producto.argmax()
producto_mas_dinero = productos[producto_mas_dinero_idx]

# Unidades totales vendidas por cada producto (sumando filas)
unidades_por_producto = ventas.sum(axis=0)
producto_mas_unidades_idx = unidades_por_producto.argmax()
producto_mas_unidades = productos[producto_mas_unidades_idx]

# --- Resultados ---
print("a) Ingreso total por producto:")
for prod, ing in zip(productos, ingreso_por_producto):
    print(f"   - {prod}: ${ing:.2f}")

print("\nb) Porcentaje del ingreso total por producto:")
for prod, pct in zip(productos, porcentaje_ingresos_redondeado):
    print(f"   - {prod}: {pct}%")

print(f"\nc) Producto que genera más dinero: {producto_mas_dinero}")
print(f"   Producto que vende más unidades: {producto_mas_unidades}")

a) Ingreso total por producto:
   - café: $1483.50
   - latte: $2018.80
   - té: $457.20
   - croissant: $1114.00
   - sándwich: $1269.00

b) Porcentaje del ingreso total por producto:
   - café: 23.4%
   - latte: 31.8%
   - té: 7.2%
   - croissant: 17.6%
   - sándwich: 20.0%

c) Producto que genera más dinero: latte
   Producto que vende más unidades: café


✍️ **Pregunta:** ¿por qué el producto que más se vende no es el que más dinero deja? Si fueras la dueña, ¿qué harías con esta información?

✍️ **Tu respuesta:**

El cafe, al ser un producto mas economico, vende mas unidades; en cambio, el sandwich y/o el latte

---
## Parte 2 · Las propinas 💵 (pandas)

Un camarero anotó, durante varias semanas, cada mesa que atendió: cuánto pagaron, cuánta propina dejaron y algunos datos de la mesa. Queremos responder una pregunta: **¿de qué depende la propina?**

El archivo es `data/tips.csv` y tiene estas columnas:

| Columna | Significado |
|---|---|
| `total_bill` | importe de la cuenta, en $ |
| `tip` | propina, en $ |
| `sex` | sexo de quien pagó (`Female` / `Male`) |
| `smoker` | si en la mesa había fumadores (`Yes` / `No`) |
| `day` | día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | `Lunch` (comida) o `Dinner` (cena) |
| `size` | número de personas en la mesa |

### Ejercicio 7 · Primer vistazo

a) Carga el archivo en un DataFrame llamado `propinas`.

b) Explóralo con `shape`, `head`, `info` y `describe`.

c) ¿Cuántas mesas hay de cada día? ¿Y de cada momento (comida/cena)?

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿cuántas mesas hay? ¿Falta algún dato? ¿Cuál fue la cuenta más alta y cuál la mesa más grande?

✍️ **Tu respuesta:**

### Ejercicio 8 · Traducir los datos

Para trabajar más cómodos, vamos a pasar todo a español.

a) Renombra las columnas: `total_bill` → `cuenta`, `tip` → `propina`, `sex` → `sexo`, `smoker` → `fumador`, `day` → `dia`, `time` → `momento`, `size` → `personas`.

b) Traduce los valores:
- `dia`: `Thur` → `Jueves`, `Fri` → `Viernes`, `Sat` → `Sábado`, `Sun` → `Domingo`
- `momento`: `Lunch` → `Comida`, `Dinner` → `Cena`
- `sexo`: `Female` → `Mujer`, `Male` → `Hombre`
- `fumador`: `Yes` → `sí`, `No` → `no`

Comprueba el resultado con `head()`.

> 💡 **Pista:** `rename(columns={...})` y `.map({...})`.

In [ ]:
# Tu código aquí

### Ejercicio 9 · Nuevas columnas

Una propina de 5 $ no significa lo mismo en una cuenta de 15 $ que en una de 50 $. Por eso vamos a crear variables más útiles:

a) `pct_propina`: la propina como **porcentaje** de la cuenta.

b) `cuenta_por_persona`: lo que pagó cada persona de la mesa.

c) `tamaño_mesa`: agrupa `personas` en `"1-2"`, `"3-4"` y `"5-6"`.

d) `generosa`: `"sí"` si la propina es del 20% o más, `"no"` en caso contrario.

> 💡 **Pistas:** operaciones entre columnas, `pd.cut` (con `bins=[0, 2, 4, 6]`) y `np.where`.

In [ ]:
# Tu código aquí

### Ejercicio 10 · Filtrar y ordenar

a) ¿Cuántas mesas dejaron una propina **generosa** (20% o más)? ¿Qué **proporción** del total representan?

b) ¿Cuántas **cenas de fin de semana** (sábado o domingo) fueron de **4 personas o más**? ¿Cuál fue su porcentaje de propina medio?

c) Muestra las **5 mesas** con mayor `pct_propina`, con las columnas `cuenta`, `propina`, `pct_propina`, `personas` y `dia`.

d) Usa `query` para obtener las mesas con **fumadores** y una cuenta **entre 10 y 20 $**. ¿Cuántas son?

> 💡 **Pistas:** filtros con `&`, `isin` y `sort_values`. Dentro de `query` el texto va entre comillas simples: `"fumador == 'sí'"`.

In [ ]:
# Tu código aquí

### Ejercicio 11 · Agrupar y resumir

a) Para cada **día**, calcula el número de mesas, la **media** y la **mediana** de `pct_propina`.

b) Para cada **momento** (comida/cena), crea un resumen con columnas con nombre: `n_mesas`, `cuenta_media`, `propina_media` y `pct_medio`.

c) Para cada número de **personas**, calcula el número de mesas y el `pct_propina` medio.

d) Crea una tabla dinámica con el `pct_propina` medio por **sexo** (filas) y **fumador** (columnas).

> 💡 **Pistas:** `groupby(...).agg([...])`, `agg(nombre=("columna", "función"))` y `pd.pivot_table`. Redondea con `.round(1)` para que se lea mejor.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** en c), ¿las mesas más grandes dejan más o menos porcentaje de propina? ¿Te fiarías de los resultados para 1, 5 o 6 personas? ¿Por qué?

✍️ **Tu respuesta:**

### Ejercicio 12 · Cuidado con las comparaciones

Haz una tabla cruzada que cuente cuántas mesas hay de cada **día** (filas) en cada **momento** (columnas).

> 💡 **Pista:** `pd.crosstab`.

In [ ]:
# Tu código aquí

✍️ **Pregunta:** si comparas las propinas de la comida con las de la cena, ¿estás comparando solo el momento del día, o también otra cosa? ¿Por qué es importante tenerlo en cuenta antes de sacar conclusiones?

✍️ **Tu respuesta:**

### Ejercicio 13 · Combinar tablas

Tenemos otra tabla con información de cada día:
```python
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})
```

a) Añade esta información a `propinas` con `merge`. Comprueba que el número de filas **no cambia**.

b) Compara la **cuenta media** y el **`pct_propina` medio** entre fin de semana y entre semana.

c) Calcula cuánto se facturó en total **cada día**, ordenado cronológicamente (de jueves a domingo).

> 💡 **Pista:** para c), agrupa por `["orden", "dia"]`: `groupby` ordena los grupos por la primera columna.

In [ ]:
# Tu código aquí

### Ejercicio 14 · De pandas a NumPy

Los modelos de Machine Learning trabajan con arrays de NumPy, así que es habitual pasar de un DataFrame a un array.

a) Crea `X` con las columnas `cuenta`, `propina` y `personas` convertidas a array de NumPy. ¿Qué forma tiene?

b) **Solo con NumPy**, calcula el porcentaje de propina de cada mesa a partir de `X`. Comprueba que su media coincide con la de `propinas["pct_propina"]`.

c) **Solo con NumPy**, ¿cuántas mesas de **2 personas** dejaron **más de 4 $** de propina?

> 💡 **Pistas:** `.to_numpy()`. En `X`, la columna 0 es la cuenta, la 1 la propina y la 2 las personas.

In [ ]:
# Tu código aquí

### ✍️ Conclusiones de la Parte 2

Imagina que el camarero te pregunta: *"¿qué tengo que hacer para conseguir mejores propinas?"*. Escribe:

1. **Tres hallazgos** sobre las propinas, cada uno apoyado en un número concreto de tus resultados.
2. **Una limitación** de estos datos que haga que debas tener cuidado con tus conclusiones.

✍️ **Tu respuesta:**

---
## Parte 3 · Visualización 📊 (matplotlib y seaborn)

Ahora te toca **elegir el gráfico adecuado** para cada pregunta. Esta parte **no tiene pistas**: decide tú qué tipo de gráfico usar y si lo haces con matplotlib o con seaborn.

Todos tus gráficos deben tener **título**, **etiquetas en los ejes** y **leyenda** cuando haya varias series o colores.

### Con los datos de la cafetería ☕

**Ejercicio 15.** Quiero ver cómo evolucionó el ingreso diario a lo largo del mes.

In [ ]:
# Tu código aquí

**Ejercicio 16.** Quiero comparar cuánto dinero generó cada producto en el mes.

In [ ]:
# Tu código aquí

**Ejercicio 17.** Quiero ver en un mismo gráfico las unidades vendidas cada día de los 5 productos, una línea por producto.

In [ ]:
# Tu código aquí

**Ejercicio 18.** Quiero ver la matriz de ventas (días × productos) como un mapa de calor.

In [ ]:
# Tu código aquí

### Con los datos de las propinas 💵

Usa el DataFrame `propinas` tal como quedó al final de la Parte 2.

**Ejercicio 19.** Quiero ver el histograma del importe de la cuenta.

In [ ]:
# Tu código aquí

**Ejercicio 20.** Quiero visualizar la relación entre la cuenta y la propina.

In [ ]:
# Tu código aquí

**Ejercicio 21.** Quiero ver esa misma relación, pero distinguiendo las comidas de las cenas.

In [ ]:
# Tu código aquí

**Ejercicio 22.** Quiero ver si hay valores atípicos en el porcentaje de propina de cada día de la semana.

In [ ]:
# Tu código aquí

**Ejercicio 23.** Quiero ver cuántas mesas se atendieron cada día.

In [ ]:
# Tu código aquí

**Ejercicio 24.** Quiero comparar la propina media según el número de personas en la mesa, separando mesas de fumadores y de no fumadores.

In [ ]:
# Tu código aquí

### Con un dataset nuevo: los pingüinos 🐧

El archivo `data/penguins.csv` tiene medidas de 344 pingüinos de tres especies de la Antártida:

| Columna | Significado |
|---|---|
| `species` | especie: `Adelie`, `Chinstrap`, `Gentoo` |
| `island` | isla donde se registró |
| `bill_length_mm` | longitud del pico (mm) |
| `bill_depth_mm` | profundidad (altura) del pico (mm) |
| `flipper_length_mm` | longitud de la aleta (mm) |
| `body_mass_g` | peso (g) |
| `sex` | `Male` / `Female` |

**Ejercicio 25.** Carga los datos en un DataFrame llamado `pinguinos`. Quiero ver el histograma del peso de los pingüinos.

In [ ]:
# Tu código aquí

**Ejercicio 26.** Quiero ver si la distribución del peso cambia según la especie.

In [ ]:
# Tu código aquí

**Ejercicio 27.** Quiero ver si hay valores atípicos en la longitud de la aleta de cada especie.

In [ ]:
# Tu código aquí

**Ejercicio 28.** Quiero visualizar la relación entre la longitud y la profundidad del pico, distinguiendo las especies.

In [ ]:
# Tu código aquí

**Ejercicio 29.** Quiero ver en una sola figura el histograma de las cuatro medidas numéricas, cada una en su propio gráfico.

In [ ]:
# Tu código aquí

**Ejercicio 30.** Quiero ver qué medidas numéricas están más relacionadas entre sí.

In [ ]:
# Tu código aquí

**Ejercicio 31.** Quiero comparar el peso medio de cada especie, separando machos y hembras.

In [ ]:
# Tu código aquí